# Phase 2 — EfficientNet-B0 Fine-tuning on FaceForensics

## Mục tiêu

Phase 1 đã dùng để **screening architecture** giữa ResNet50, EfficientNet-B0 và ConvNeXt-Tiny. Phase 2 chỉ tập trung vào **EfficientNet-B0**.

**Checkpoint đầu vào mặc định:**
`efficientnet_b0_seed123.pt`

**Mục tiêu đánh giá:**
- Accuracy ≥ 95%
- F1-score ≥ 94%
- ROC-AUC ≥ 0.94
- Compression Robustness Accuracy ≥ 90% ở JPEG Quality 40–60

> Các ngưỡng trên là **target**, không phải kết quả được đảm bảo. Nếu validation không đạt, notebook giữ lại kết quả trung thực thay vì ép model bằng cách tăng epoch vô hạn.

## Protocol Phase 2

### Stage A — Head adaptation
- Load checkpoint Phase 1.
- Freeze backbone.
- Chỉ train classification head.
- 2 epochs.
- Head LR = 1e-4.

### Stage B — Full fine-tuning
- Unfreeze toàn bộ EfficientNet-B0.
- Backbone LR = 1e-5.
- Head LR = 1e-4.
- 12 epochs tối đa.
- Warmup + cosine decay.
- Early stopping theo validation F1.

### Data
- Full training set được sử dụng trong mỗi epoch.
- Validation dùng một subset cố định, cân bằng REAL/FAKE, để so sánh nhất quán.
- Test set **không được dùng trong model selection**.

### Augmentation Phase 2
1. Resize 224×224.
2. Random Horizontal Flip, p=0.5.
3. Mild ColorJitter.
4. Random JPEG Q50–80, p=0.30.
5. Mild Gaussian Blur, p=0.10.
6. Mild Gaussian Noise, p=0.05.
7. ImageNet normalization.

Không dùng rotation/crop/erasing/mixup/cutmix mạnh ở giai đoạn này vì các phép biến đổi đó có thể làm mất các artifact khuôn mặt mà deepfake detector cần học.

### Compression robustness
Đánh giá deterministic trên cùng validation subset:
- HQ
- JPEG Q80
- JPEG Q60
- JPEG Q40

### Checkpoint selection
Score validation:

S = 0.40 × F1(HQ)
  + 0.30 × ROC-AUC(HQ)
  + 0.20 × F1(Q40)
  + 0.10 × Accuracy(Q40)

Threshold được tối ưu **chỉ trên validation** theo F1.

## Hai bảng chính
- **Table 1:** metrics theo epoch của quá trình fine-tuning.
- **Table 2:** kết quả validation cuối cùng của checkpoint tốt nhất trên HQ/Q80/Q60/Q40.

Cell test cuối cùng chỉ được chạy sau khi Phase 2 đã khóa checkpoint + threshold.


In [ ]:
# ============================================================
# CELL 01 — INSTALL / IMPORTS
# Làm gì:
#   - Cài timm và import toàn bộ thư viện.
# Kỹ thuật:
#   - PyTorch + timm + AMP FP16 + sklearn.
# ============================================================

!pip -q install -U timm

import os
import gc
import io
import json
import math
import time
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms

import timm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("timm:", timm.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# CELL 02 — T4 CONFIGURATION
# Làm gì:
#   - Đặt hyperparameters của Phase 2.
# Kỹ thuật:
#   - Full-data fine-tuning nhưng vẫn giới hạn memory/time.
#   - AMP + channels_last cho T4.
# ============================================================

SEED = 123

MODEL_NAME = "efficientnet_b0"

IMAGE_SIZE = 224

# Full training set.
BATCH_SIZE = 64
NUM_WORKERS = 2

# Stage A: classifier head.
STAGE_A_EPOCHS = 2
HEAD_LR_STAGE_A = 1e-4

# Stage B: all layers.
STAGE_B_MAX_EPOCHS = 12
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4

WEIGHT_DECAY = 1e-4

# Warmup.
WARMUP_EPOCHS = 2

# Early stopping.
PATIENCE = 3
MIN_DELTA = 1e-4

# Balanced validation subset.
VAL_SUBSET_SIZE = 2_000
VAL_PER_CLASS = VAL_SUBSET_SIZE // 2
VAL_SELECTION_SEED = 2026

# JPEG robustness.
ROBUSTNESS_QUALITIES = [80, 60, 40]

# Phase-2 augmentation.
JPEG_AUG_P = 0.30
JPEG_AUG_MIN = 50
JPEG_AUG_MAX = 80

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = DEVICE.type == "cuda"

CLASS_NAMES = ["REAL", "FAKE"]
NUM_CLASSES = 2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

OUTPUT_DIR = Path("/content/phase2_results")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("DEVICE:", DEVICE)
print("MODEL:", MODEL_NAME)
print("INPUT:", IMAGE_SIZE, "x", IMAGE_SIZE)
print("BATCH:", BATCH_SIZE)
print("Stage A:", STAGE_A_EPOCHS, "epochs")
print("Stage B:", STAGE_B_MAX_EPOCHS, "epochs max")
print("SEED:", SEED)


In [ ]:
# ============================================================
# CELL 03 — MOUNT GOOGLE DRIVE
# Làm gì:
#   - Mount Drive.
#   - Cho phép lấy dataset và lưu checkpoint Phase 2 lâu dài.
# Kỹ thuật:
#   - Dataset nên được extract vào /content để training nhanh.
#   - Checkpoint cuối được copy sang Drive.
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)

print("Google Drive mounted.")


In [ ]:
# ============================================================
# CELL 04 — DATASET + PHASE 1 CHECKPOINT PATH
# Làm gì:
#   - Xác định dataset root.
#   - Tìm checkpoint EfficientNet-B0 seed=123 từ Phase 1.
#
# Kỹ thuật:
#   - Có nhiều fallback path để notebook chạy được nếu Phase 1
#     vẫn còn trong runtime hoặc đã được copy sang Drive.
# ============================================================

# ---------- DATASET ----------
DATA_ROOT_CANDIDATES = [
    Path("/content/face_frames"),
    Path("/content/face_frames/face_frames"),
]

DATA_ROOT = None

for candidate in DATA_ROOT_CANDIDATES:
    if all(
        (candidate / split).exists()
        for split in ["train", "val", "test"]
    ):
        DATA_ROOT = candidate
        break

# Nếu chưa tìm được, tìm một cấp dưới /content.
if DATA_ROOT is None:
    for p in Path("/content").iterdir():
        if not p.is_dir():
            continue

        if all(
            (p / split).exists()
            for split in ["train", "val", "test"]
        ):
            DATA_ROOT = p
            break

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy DATA_ROOT có train/val/test. "
        "Hãy sửa DATA_ROOT trong Cell 04."
    )

# ---------- CHECKPOINT ----------
CHECKPOINT_CANDIDATES = [
    Path(
        "/content/phase1_results/checkpoints/"
        "efficientnet_b0_seed123.pt"
    ),
    Path(
        "/content/phase1_results/selected_backbone/"
        "efficientnet_b0_best_for_finetuning.pt"
    ),
    Path(
        "/content/drive/MyDrive/phase1_results/checkpoints/"
        "efficientnet_b0_seed123.pt"
    ),
    Path(
        "/content/drive/MyDrive/phase1_results/selected_backbone/"
        "efficientnet_b0_best_for_finetuning.pt"
    ),
]

CHECKPOINT_PATH = None

for candidate in CHECKPOINT_CANDIDATES:
    if candidate.exists():
        CHECKPOINT_PATH = candidate
        break

if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy checkpoint EfficientNet-B0 seed=123. "
        "Hãy copy checkpoint Phase 1 vào /content/phase1_results/checkpoints/ "
        "hoặc sửa CHECKPOINT_CANDIDATES."
    )

print("DATA_ROOT:", DATA_ROOT)
print("CHECKPOINT:", CHECKPOINT_PATH)


In [ ]:
# ============================================================
# CELL 05 — DATASET SANITY CHECK
# Làm gì:
#   - Kiểm tra số lượng ảnh và REAL/FAKE ratio.
# Kỹ thuật:
#   - Data integrity check trước fine-tuning.
# ============================================================

VALID_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
}

def count_images(split_dir):
    result = {}

    for cls in CLASS_NAMES:
        folder = split_dir / cls

        if not folder.exists():
            raise FileNotFoundError(
                f"Missing: {folder}"
            )

        result[cls] = sum(
            1
            for p in folder.rglob("*")
            if (
                p.is_file()
                and p.suffix.lower() in VALID_EXTS
            )
        )

    return result


counts = {}

for split in ["train", "val", "test"]:
    c = count_images(
        DATA_ROOT / split
    )

    counts[split] = c

    print(
        f"{split:>5}: "
        f"REAL={c['REAL']:,} | "
        f"FAKE={c['FAKE']:,} | "
        f"TOTAL={sum(c.values()):,}"
    )

train_ratio = (
    counts["train"]["FAKE"]
    / max(counts["train"]["REAL"], 1)
)

print(
    f"Train FAKE/REAL ratio: {train_ratio:.2f}:1"
)


In [ ]:
# ============================================================
# CELL 06 — PHASE-2 AUGMENTATION + EVALUATION TRANSFORMS
# Làm gì:
#   - Tạo augmentation nhẹ hơn Phase 1.
#   - Input tăng lên 224x224.
#   - Không dùng random augmentation ở validation/test.
#
# Kỹ thuật:
#   - Moderate augmentation.
#   - JPEG Q50–80 trong training để tăng robustness.
# ============================================================

class RandomJPEGCompression:
    def __init__(
        self,
        p=0.30,
        min_quality=50,
        max_quality=80,
    ):
        self.p = p
        self.min_quality = min_quality
        self.max_quality = max_quality

    def __call__(self, img):
        if random.random() > self.p:
            return img

        quality = random.randint(
            self.min_quality,
            self.max_quality,
        )

        img = img.convert("RGB")

        buffer = io.BytesIO()

        img.save(
            buffer,
            format="JPEG",
            quality=quality,
        )

        buffer.seek(0)

        with Image.open(buffer) as compressed:
            return compressed.convert("RGB").copy()


class RandomGaussianNoise:
    def __init__(
        self,
        p=0.05,
        std=0.008,
    ):
        self.p = p
        self.std = std

    def __call__(self, tensor):
        if random.random() > self.p:
            return tensor

        noise = torch.randn_like(tensor) * self.std

        return torch.clamp(
            tensor + noise,
            0.0,
            1.0,
        )


train_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.RandomHorizontalFlip(
        p=0.50
    ),

    transforms.ColorJitter(
        brightness=0.08,
        contrast=0.08,
        saturation=0.06,
        hue=0.015,
    ),

    RandomJPEGCompression(
        p=JPEG_AUG_P,
        min_quality=JPEG_AUG_MIN,
        max_quality=JPEG_AUG_MAX,
    ),

    transforms.RandomApply(
        [
            transforms.GaussianBlur(
                kernel_size=3,
                sigma=(0.1, 0.6),
            )
        ],
        p=0.10,
    ),

    transforms.ToTensor(),

    RandomGaussianNoise(
        p=0.05,
        std=0.008,
    ),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD,
    ),
])


eval_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD,
    ),
])

print("Phase-2 augmentation:")
print("  Resize:", IMAGE_SIZE)
print("  HorizontalFlip: p=0.50")
print("  Mild ColorJitter")
print("  JPEG augmentation: Q50–80, p=0.30")
print("  GaussianBlur: p=0.10")
print("  GaussianNoise: p=0.05")


In [ ]:
# ============================================================
# CELL 07 — DATASET CLASS
# Làm gì:
#   - Lazy-load ảnh từ disk.
# Kỹ thuật:
#   - Không load toàn bộ dataset vào RAM.
#   - PIL -> tensor only when a sample is requested.
# ============================================================

class FaceFolderDataset(Dataset):
    def __init__(
        self,
        root,
        transform=None,
    ):
        self.root = Path(root)
        self.transform = transform
        self.samples = []

        for label, cls in enumerate(CLASS_NAMES):
            folder = self.root / cls

            for p in sorted(
                folder.rglob("*")
            ):
                if (
                    p.is_file()
                    and p.suffix.lower()
                    in VALID_EXTS
                ):
                    self.samples.append(
                        (str(p), label)
                    )

        if not self.samples:
            raise RuntimeError(
                f"No images found in {self.root}"
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        with Image.open(path) as img:
            img = img.convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        return img, label


train_dataset = FaceFolderDataset(
    DATA_ROOT / "train",
    transform=train_transform,
)

val_dataset = FaceFolderDataset(
    DATA_ROOT / "val",
    transform=eval_transform,
)

test_dataset = FaceFolderDataset(
    DATA_ROOT / "test",
    transform=eval_transform,
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))


In [ ]:
# ============================================================
# CELL 08 — FIXED BALANCED VALIDATION SUBSET
# Làm gì:
#   - Chọn 1,000 REAL + 1,000 FAKE cố định.
#   - Dùng cùng subset cho toàn bộ Phase 2.
# Kỹ thuật:
#   - Balanced validation để F1/Accuracy phản ánh hai class
#     công bằng hơn trong quá trình model selection.
# ============================================================

val_labels = np.array([
    label
    for _, label in val_dataset.samples
])

real_idx = np.where(
    val_labels == 0
)[0]

fake_idx = np.where(
    val_labels == 1
)[0]

if (
    len(real_idx) < VAL_PER_CLASS
    or len(fake_idx) < VAL_PER_CLASS
):
    raise ValueError(
        "Validation không đủ REAL/FAKE "
        "cho balanced subset."
    )

rng = np.random.default_rng(
    VAL_SELECTION_SEED
)

val_real = rng.choice(
    real_idx,
    size=VAL_PER_CLASS,
    replace=False,
)

val_fake = rng.choice(
    fake_idx,
    size=VAL_PER_CLASS,
    replace=False,
)

VAL_INDICES = np.concatenate([
    val_real,
    val_fake,
]).tolist()

val_subset = Subset(
    val_dataset,
    VAL_INDICES,
)

print(
    "Fixed validation subset:",
    len(val_subset),
)
print(
    "REAL:",
    VAL_PER_CLASS,
    "| FAKE:",
    VAL_PER_CLASS,
)


In [ ]:
# ============================================================
# CELL 09 — DETERMINISTIC COMPRESSION DATASETS
# Làm gì:
#   - Tạo HQ/Q80/Q60/Q40 validation views.
#   - Không lưu compressed images xuống disk.
# Kỹ thuật:
#   - On-the-fly corruption benchmark.
#   - Cùng image indices ở mọi compression level.
# ============================================================

class JPEGSubsetDataset(Dataset):
    def __init__(
        self,
        base_dataset,
        indices,
        quality,
    ):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.quality = quality

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        base_idx = self.indices[idx]

        path, label = (
            self.base_dataset.samples[
                base_idx
            ]
        )

        with Image.open(path) as img:
            img = img.convert("RGB")

        buffer = io.BytesIO()

        img.save(
            buffer,
            format="JPEG",
            quality=self.quality,
        )

        buffer.seek(0)

        with Image.open(buffer) as compressed:
            img = compressed.convert(
                "RGB"
            ).copy()

        img = eval_transform(img)

        return img, label


compression_datasets = {}

for q in ROBUSTNESS_QUALITIES:
    compression_datasets[q] = (
        JPEGSubsetDataset(
            val_dataset,
            VAL_INDICES,
            quality=q,
        )
    )

print(
    "Compression levels:",
    ROBUSTNESS_QUALITIES,
)


In [ ]:
# ============================================================
# CELL 10 — REPRODUCIBILITY + DATALOADERS
# Làm gì:
#   - Seed tất cả RNG.
#   - Full training set mỗi epoch.
#   - WeightedRandomSampler cân bằng REAL/FAKE.
# Kỹ thuật:
#   - Weighted sampling thay vì giảm dataset FAKE.
#   - pin_memory + non_blocking + persistent workers.
# ============================================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Tăng throughput trên T4 với input shape cố định.
    torch.backends.cudnn.benchmark = True


seed_everything(SEED)


def make_balanced_sampler(
    dataset,
    seed,
):
    labels = np.array([
        label
        for _, label in dataset.samples
    ])

    counts = np.bincount(
        labels,
        minlength=NUM_CLASSES,
    )

    class_weights = (
        1.0
        / np.maximum(
            counts,
            1,
        )
    )

    sample_weights = class_weights[
        labels
    ]

    generator = torch.Generator()

    generator.manual_seed(seed)

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(
            sample_weights,
            dtype=torch.double,
        ),
        num_samples=len(dataset),
        replacement=True,
        generator=generator,
    )

    return sampler


def make_loader(
    dataset,
    sampler=None,
    shuffle=False,
):
    kwargs = {
        "dataset": dataset,
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pin_memory": DEVICE.type == "cuda",
        "persistent_workers": NUM_WORKERS > 0,
    }

    if sampler is not None:
        kwargs["sampler"] = sampler
    else:
        kwargs["shuffle"] = shuffle

    return DataLoader(
        **kwargs
    )


train_sampler = make_balanced_sampler(
    train_dataset,
    SEED,
)

train_loader = make_loader(
    train_dataset,
    sampler=train_sampler,
)

val_loader = make_loader(
    val_subset,
    shuffle=False,
)

compression_loaders = {
    q: make_loader(
        dataset,
        shuffle=False,
    )
    for q, dataset
    in compression_datasets.items()
}

print(
    "Training batches/epoch:",
    math.ceil(
        len(train_dataset)
        / BATCH_SIZE
    ),
)


In [ ]:
# ============================================================
# CELL 11 — LOAD + VERIFY EFFICIENTNET-B0 CHECKPOINT
# Làm gì:
#   - Load checkpoint Phase 1 seed=123.
#   - Verify architecture và metadata.
# Kỹ thuật:
#   - Transfer learning from Phase 1.
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

print(
    "Checkpoint keys:",
    list(checkpoint.keys()),
)

if (
    "model_name" in checkpoint
    and checkpoint["model_name"]
    != MODEL_NAME
):
    raise ValueError(
        "Checkpoint không phải EfficientNet-B0."
    )

if (
    "seed" in checkpoint
    and int(checkpoint["seed"]) != SEED
):
    raise ValueError(
        "Checkpoint không phải seed=123."
    )

model = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=NUM_CLASSES,
)

state_dict = checkpoint.get(
    "model_state_dict",
    checkpoint,
)

model.load_state_dict(
    state_dict,
    strict=True,
)

model = model.to(DEVICE)

if DEVICE.type == "cuda":
    model = model.to(
        memory_format=torch.channels_last
    )

print(
    "Loaded:",
    MODEL_NAME,
    "| seed:",
    checkpoint.get("seed", "unknown"),
    "| source epoch:",
    checkpoint.get("epoch", "unknown"),
)


In [ ]:
# ============================================================
# CELL 12 — VERIFY CLASSIFIER + LOSS
# Làm gì:
#   - Xác định classifier của EfficientNet-B0.
#   - Dùng weighted CrossEntropyLoss.
# Kỹ thuật:
#   - Class weighting giảm bias do REAL:FAKE ≈ 1:6.
#   - Không oversample bằng cách xóa dữ liệu FAKE.
# ============================================================

print(
    "Classifier:",
    model.classifier,
)

# Xác định output layer cuối của classifier.
if isinstance(
    model.classifier,
    nn.Sequential,
):
    head_layer = model.classifier[-1]
else:
    head_layer = model.classifier

if not isinstance(
    head_layer,
    nn.Linear,
):
    raise TypeError(
        "Không xác định được Linear classifier "
        "của EfficientNet-B0."
    )

# Class weight tính từ full training distribution.
train_labels = np.array([
    label
    for _, label in train_dataset.samples
])

class_counts = np.bincount(
    train_labels,
    minlength=NUM_CLASSES,
)

N = class_counts.sum()

class_weights = (
    N
    / (
        NUM_CLASSES
        * np.maximum(
            class_counts,
            1,
        )
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE,
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

print(
    "Class counts:",
    class_counts.tolist(),
)

print(
    "Class weights:",
    class_weights.detach().cpu().numpy()
)


In [ ]:
# ============================================================
# CELL 13 — METRICS + THRESHOLD OPTIMIZATION
# Làm gì:
#   - Tính Accuracy, Precision, Recall, F1, ROC-AUC.
#   - Tìm threshold tốt nhất trên validation.
# Kỹ thuật:
#   - Threshold selection chỉ dùng validation.
#   - Không tối ưu threshold trên test.
# ============================================================

def compute_metrics(
    y_true,
    y_prob,
    threshold=0.5,
):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "auc": roc_auc_score(
            y_true,
            y_prob,
        ),
    }


@torch.no_grad()
def predict_loader(
    model,
    loader,
):
    model.eval()

    labels_all = []
    probs_all = []

    for images, labels in loader:

        if DEVICE.type == "cuda":
            images = images.to(
                DEVICE,
                non_blocking=True,
                memory_format=torch.channels_last,
            )
        else:
            images = images.to(
                DEVICE,
                non_blocking=True,
            )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            logits = model(images)

        probs = torch.softmax(
            logits,
            dim=1,
        )[:, 1]

        labels_all.extend(
            labels.numpy().tolist()
        )

        probs_all.extend(
            probs.float()
            .cpu()
            .numpy()
            .tolist()
        )

    return (
        np.asarray(labels_all),
        np.asarray(probs_all),
    )


def find_best_threshold(
    y_true,
    y_prob,
    low=0.20,
    high=0.80,
    step=0.01,
):
    thresholds = np.arange(
        low,
        high + 1e-9,
        step,
    )

    rows = []

    for threshold in thresholds:
        metrics = compute_metrics(
            y_true,
            y_prob,
            threshold=threshold,
        )

        rows.append({
            "threshold": threshold,
            **metrics,
        })

    result = pd.DataFrame(rows)

    best = result.sort_values(
        [
            "f1",
            "accuracy",
            "precision",
        ],
        ascending=False,
    ).iloc[0]

    return (
        float(best["threshold"]),
        result,
    )


In [ ]:
# ============================================================
# CELL 14 — MODEL FREEZE / UNFREEZE HELPERS
# Làm gì:
#   - Stage A: freeze toàn bộ backbone, chỉ train classifier.
#   - Stage B: unfreeze toàn bộ model.
# Kỹ thuật:
#   - Discriminative learning rates.
#   - Head adaptation trước rồi mới fine-tune representation.
# ============================================================

def freeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = False

    for param in model.classifier.parameters():
        param.requires_grad = True


def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True


def count_trainable_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


print(
    "Total parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    ) / 1e6,
    "M",
)


In [ ]:
# ============================================================
# CELL 15 — STAGE A: HEAD ADAPTATION
# Làm gì:
#   - Freeze EfficientNet-B0 backbone.
#   - Train classifier 2 epochs.
#   - Đánh giá validation sau mỗi epoch.
# Kỹ thuật:
#   - Low-risk adaptation trước full fine-tuning.
# ============================================================

freeze_backbone(model)

print(
    "Trainable parameters in Stage A:",
    count_trainable_parameters(model) / 1e6,
    "M",
)

optimizer_A = torch.optim.AdamW(
    [
        {
            "params": model.classifier.parameters(),
            "lr": HEAD_LR_STAGE_A,
        }
    ],
    weight_decay=WEIGHT_DECAY,
)

history = []


def run_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
):
    model.train()

    running_loss = 0.0
    n_batches = 0

    for images, labels in loader:

        if DEVICE.type == "cuda":
            images = images.to(
                DEVICE,
                non_blocking=True,
                memory_format=torch.channels_last,
            )
        else:
            images = images.to(
                DEVICE,
                non_blocking=True,
            )

        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            logits = model(images)
            loss = criterion(
                logits,
                labels,
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        n_batches += 1

    return (
        running_loss
        / max(n_batches, 1)
    )


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)

global_epoch = 0

for stage_epoch in range(
    1,
    STAGE_A_EPOCHS + 1,
):
    global_epoch += 1

    start = time.time()

    train_loss = run_epoch(
        model,
        train_loader,
        optimizer_A,
        criterion,
        scaler,
    )

    y_true, y_prob = predict_loader(
        model,
        val_loader,
    )

    threshold, _ = find_best_threshold(
        y_true,
        y_prob,
    )

    val_metrics = compute_metrics(
        y_true,
        y_prob,
        threshold,
    )

    elapsed = time.time() - start

    history.append({
        "stage": "A_head",
        "epoch": global_epoch,
        "stage_epoch": stage_epoch,
        "loss": train_loss,
        "threshold": threshold,
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_auc": val_metrics["auc"],
        "time_sec": elapsed,
    })

    print(
        f"Stage A | Epoch "
        f"{stage_epoch}/{STAGE_A_EPOCHS} | "
        f"Loss={train_loss:.4f} | "
        f"F1={val_metrics['f1']:.4f} | "
        f"AUC={val_metrics['auc']:.4f} | "
        f"Threshold={threshold:.2f} | "
        f"Time={elapsed:.1f}s"
    )


In [ ]:
# ============================================================
# CELL 16 — STAGE B: FULL FINE-TUNING
# Làm gì:
#   - Unfreeze toàn bộ EfficientNet-B0.
#   - Backbone LR = 1e-5.
#   - Head LR = 1e-4.
#   - Warmup + cosine decay.
#   - Early stopping theo validation F1.
#
# Kỹ thuật:
#   - Differential learning rates.
#   - Warmup tránh update quá mạnh ngay khi mở backbone.
#   - Cosine decay giúp fine-tuning ổn định.
# ============================================================

unfreeze_all(model)

print(
    "Trainable parameters in Stage B:",
    count_trainable_parameters(model) / 1e6,
    "M",
)

optimizer_B = torch.optim.AdamW(
    [
        {
            "params": [
                p
                for name, p
                in model.named_parameters()
                if (
                    p.requires_grad
                    and not name.startswith(
                        "classifier."
                    )
                )
            ],
            "lr": BACKBONE_LR,
        },
        {
            "params": model.classifier.parameters(),
            "lr": HEAD_LR,
        },
    ],
    weight_decay=WEIGHT_DECAY,
)


total_stage_b_epochs = (
    STAGE_B_MAX_EPOCHS
)

def lr_lambda(epoch):
    # epoch bắt đầu từ 0.
    if epoch < WARMUP_EPOCHS:
        return float(
            epoch + 1
        ) / max(
            WARMUP_EPOCHS,
            1,
        )

    progress = (
        epoch - WARMUP_EPOCHS
    ) / max(
        total_stage_b_epochs
        - WARMUP_EPOCHS,
        1,
    )

    return 0.5 * (
        1.0
        + math.cos(
            math.pi
            * progress
        )
    )


scheduler_B = torch.optim.lr_scheduler.LambdaLR(
    optimizer_B,
    lr_lambda=lr_lambda,
)

best_score = -np.inf
best_epoch = None
best_threshold = 0.5
best_state = None
epochs_without_improvement = 0

for stage_epoch in range(
    1,
    STAGE_B_MAX_EPOCHS + 1,
):
    global_epoch += 1

    start = time.time()

    train_loss = run_epoch(
        model,
        train_loader,
        optimizer_B,
        criterion,
        scaler,
    )

    scheduler_B.step()

    # Clean validation.
    y_true_hq, y_prob_hq = predict_loader(
        model,
        val_loader,
    )

    threshold_hq, _ = find_best_threshold(
        y_true_hq,
        y_prob_hq,
    )

    hq_metrics = compute_metrics(
        y_true_hq,
        y_prob_hq,
        threshold_hq,
    )

    # Q40 is measured every epoch for compression-aware
    # checkpoint selection, but no threshold is optimized on Q40.
    y_true_q40, y_prob_q40 = predict_loader(
        model,
        compression_loaders[40],
    )

    q40_metrics = compute_metrics(
        y_true_q40,
        y_prob_q40,
        threshold_hq,
    )

    # Selection score specified in notebook.
    score = (
        0.40 * hq_metrics["f1"]
        + 0.30 * hq_metrics["auc"]
        + 0.20 * q40_metrics["f1"]
        + 0.10 * q40_metrics["accuracy"]
    )

    elapsed = time.time() - start

    current_lrs = [
        group["lr"]
        for group in optimizer_B.param_groups
    ]

    history.append({
        "stage": "B_full",
        "epoch": global_epoch,
        "stage_epoch": stage_epoch,
        "loss": train_loss,
        "threshold": threshold_hq,
        "val_accuracy": hq_metrics["accuracy"],
        "val_precision": hq_metrics["precision"],
        "val_recall": hq_metrics["recall"],
        "val_f1": hq_metrics["f1"],
        "val_auc": hq_metrics["auc"],
        "q40_accuracy": q40_metrics["accuracy"],
        "q40_f1": q40_metrics["f1"],
        "q40_auc": q40_metrics["auc"],
        "selection_score": score,
        "backbone_lr": current_lrs[0],
        "head_lr": current_lrs[1],
        "time_sec": elapsed,
    })

    print(
        f"Stage B | Epoch "
        f"{stage_epoch}/{STAGE_B_MAX_EPOCHS} | "
        f"Loss={train_loss:.4f} | "
        f"HQ F1={hq_metrics['f1']:.4f} | "
        f"HQ AUC={hq_metrics['auc']:.4f} | "
        f"Q40 F1={q40_metrics['f1']:.4f} | "
        f"Score={score:.4f} | "
        f"Thr={threshold_hq:.2f} | "
        f"Time={elapsed:.1f}s"
    )

    # Best checkpoint.
    if score > (
        best_score + MIN_DELTA
    ):
        best_score = score
        best_epoch = global_epoch
        best_threshold = threshold_hq
        epochs_without_improvement = 0

        best_state = {
            k: v.detach().cpu().clone()
            for k, v
            in model.state_dict().items()
        }

        torch.save(
            {
                "model_name": MODEL_NAME,
                "phase": 2,
                "seed": SEED,
                "source_checkpoint": str(
                    CHECKPOINT_PATH
                ),
                "best_epoch": best_epoch,
                "best_selection_score": best_score,
                "best_threshold": best_threshold,
                "model_state_dict": best_state,
            },
            CHECKPOINT_DIR
            / "efficientnet_b0_phase2_best.pt",
        )

    else:
        epochs_without_improvement += 1

    if (
        epochs_without_improvement
        >= PATIENCE
    ):
        print(
            f"Early stopping at Stage-B "
            f"epoch {stage_epoch}."
        )
        break

print(
    "Best global epoch:",
    best_epoch,
)

print(
    "Best validation selection score:",
    best_score,
)

print(
    "Best validation threshold:",
    best_threshold,
)


In [ ]:
# ============================================================
# CELL 17 — TABLE 1: EPOCH-LEVEL FINE-TUNING METRICS
# Làm gì:
#   - Tổng hợp quá trình Stage A + Stage B.
#   - Chỉ hiển thị một bảng epoch-level.
# Kỹ thuật:
#   - Theo dõi convergence và compression-aware score.
# ============================================================

history_df = pd.DataFrame(history)

history_df.to_csv(
    OUTPUT_DIR / "phase2_history.csv",
    index=False,
)

display_cols = [
    "stage",
    "epoch",
    "loss",
    "val_accuracy",
    "val_precision",
    "val_recall",
    "val_f1",
    "val_auc",
    "q40_accuracy",
    "q40_f1",
    "selection_score",
    "threshold",
    "time_sec",
]

table1 = history_df[
    [
        c
        for c in display_cols
        if c in history_df.columns
    ]
].copy()

for col in [
    "loss",
    "val_accuracy",
    "val_precision",
    "val_recall",
    "val_f1",
    "val_auc",
    "q40_accuracy",
    "q40_f1",
    "selection_score",
    "threshold",
]:
    if col in table1.columns:
        table1[col] = table1[
            col
        ].round(4)

table1["time_sec"] = table1[
    "time_sec"
].round(1)

table1.to_csv(
    OUTPUT_DIR
    / "TABLE_1_phase2_epoch_metrics.csv",
    index=False,
)

display(table1)


In [ ]:
# ============================================================
# CELL 18 — LOAD BEST CHECKPOINT + FINAL VALIDATION
# Làm gì:
#   - Load checkpoint tốt nhất theo validation selection score.
#   - Re-optimize threshold on HQ validation only.
#   - Evaluate HQ/Q80/Q60/Q40 using the same threshold.
# Kỹ thuật:
#   - Final validation before test.
#   - Threshold is locked after this cell.
# ============================================================

BEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "efficientnet_b0_phase2_best.pt"
)

if not BEST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "Best Phase-2 checkpoint không tồn tại."
    )

best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

best_model = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=NUM_CLASSES,
).to(DEVICE)

best_model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)

if DEVICE.type == "cuda":
    best_model = best_model.to(
        memory_format=torch.channels_last
    )

# Optimize threshold ONLY on HQ validation.
y_true_hq, y_prob_hq = predict_loader(
    best_model,
    val_loader,
)

locked_threshold, threshold_curve = (
    find_best_threshold(
        y_true_hq,
        y_prob_hq,
    )
)

final_validation_rows = []

# HQ
hq = compute_metrics(
    y_true_hq,
    y_prob_hq,
    locked_threshold,
)

final_validation_rows.append({
    "Condition": "HQ",
    "JPEG Quality": "Original",
    "Threshold": locked_threshold,
    **hq,
})

# Compression
for q in ROBUSTNESS_QUALITIES:
    y_true_q, y_prob_q = predict_loader(
        best_model,
        compression_loaders[q],
    )

    metrics_q = compute_metrics(
        y_true_q,
        y_prob_q,
        locked_threshold,
    )

    final_validation_rows.append({
        "Condition": f"JPEG Q{q}",
        "JPEG Quality": q,
        "Threshold": locked_threshold,
        **metrics_q,
    })

final_validation_df = pd.DataFrame(
    final_validation_rows
)

final_validation_df.to_csv(
    OUTPUT_DIR
    / "phase2_final_validation.csv",
    index=False,
)

print(
    "Locked threshold:",
    round(locked_threshold, 4),
)

print(
    "Best epoch:",
    best_checkpoint["best_epoch"],
)

print(
    "Best selection score:",
    round(
        best_checkpoint[
            "best_selection_score"
        ],
        4,
    ),
)


In [ ]:
# ============================================================
# CELL 19 — TABLE 2: FINAL VALIDATION + COMPRESSION ROBUSTNESS
# Làm gì:
#   - Hiển thị kết quả cuối cùng của checkpoint tốt nhất.
#   - HQ/Q80/Q60/Q40.
#   - Chỉ bảng thứ hai của notebook.
# Kỹ thuật:
#   - Same threshold across all compression levels.
#   - Cho phép đánh giá degradation do compression.
# ============================================================

table2 = final_validation_df.copy()

# Accuracy drop relative to HQ.
hq_acc = float(
    table2.loc[
        table2["Condition"] == "HQ",
        "accuracy",
    ].iloc[0]
)

table2["Accuracy Drop vs HQ"] = (
    hq_acc - table2["accuracy"]
)

for col in [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "auc",
    "Accuracy Drop vs HQ",
]:
    table2[col] = table2[
        col
    ].round(4)

table2.to_csv(
    OUTPUT_DIR
    / "TABLE_2_phase2_final_validation.csv",
    index=False,
)

display(table2)

# Target check — informational only.
hq_row = table2[
    table2["Condition"] == "HQ"
].iloc[0]

q40_row = table2[
    table2["Condition"] == "JPEG Q40"
].iloc[0]

print()
print("TARGET CHECK — VALIDATION")
print(
    "HQ Accuracy >= 95%:",
    bool(hq_row["accuracy"] >= 0.95),
)
print(
    "HQ F1 >= 94%:",
    bool(hq_row["f1"] >= 0.94),
)
print(
    "HQ ROC-AUC >= 0.94:",
    bool(hq_row["auc"] >= 0.94),
)
print(
    "Q40 Accuracy >= 90%:",
    bool(q40_row["accuracy"] >= 0.90),
)


In [ ]:
# ============================================================
# CELL 20 — SAVE PHASE 2 ARTIFACTS + COPY TO DRIVE
# Làm gì:
#   - Lưu threshold, checkpoint và configuration.
#   - Copy checkpoint sang Google Drive để không mất sau khi
#     Colab runtime reset.
# Kỹ thuật:
#   - Artifact versioning.
# ============================================================

phase2_info = {
    "model": MODEL_NAME,
    "seed": SEED,
    "source_checkpoint": str(
        CHECKPOINT_PATH
    ),
    "best_checkpoint": str(
        BEST_CHECKPOINT_PATH
    ),
    "best_epoch": int(
        best_checkpoint["best_epoch"]
    ),
    "best_selection_score": float(
        best_checkpoint[
            "best_selection_score"
        ]
    ),
    "locked_threshold": float(
        locked_threshold
    ),
    "input_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "stage_a_epochs": STAGE_A_EPOCHS,
    "stage_b_max_epochs": STAGE_B_MAX_EPOCHS,
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "weight_decay": WEIGHT_DECAY,
    "augmentation": [
        "Resize 224x224",
        "HorizontalFlip p=0.50",
        "Mild ColorJitter",
        "JPEG Q50-Q80 p=0.30",
        "GaussianBlur p=0.10",
        "GaussianNoise p=0.05",
    ],
    "robustness_levels": [
        "HQ",
        "JPEG Q80",
        "JPEG Q60",
        "JPEG Q40",
    ],
    "test_used_for_model_selection": False,
}

with open(
    OUTPUT_DIR / "phase2_info.json",
    "w",
) as f:
    json.dump(
        phase2_info,
        f,
        indent=2,
    )

DRIVE_PHASE2_DIR = (
    Path("/content/drive/MyDrive")
    / "phase2_efficientnet_b0"
)

DRIVE_PHASE2_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy2(
    BEST_CHECKPOINT_PATH,
    DRIVE_PHASE2_DIR
    / "efficientnet_b0_phase2_best.pt",
)

shutil.copy2(
    OUTPUT_DIR / "phase2_info.json",
    DRIVE_PHASE2_DIR
    / "phase2_info.json",
)

print(
    "Phase 2 checkpoint copied to:",
    DRIVE_PHASE2_DIR,
)


In [ ]:
# ============================================================
# CELL 21 — FINAL TEST EVALUATION
# ============================================================
# QUAN TRỌNG:
#   Chỉ chạy cell này sau khi:
#   1) Phase 2 đã hoàn thành.
#   2) Checkpoint đã khóa.
#   3) Threshold đã khóa trên validation.
#
# Test không được dùng để:
#   - chọn epoch
#   - chọn checkpoint
#   - chọn threshold
#   - thay đổi hyperparameters
#
# Kỹ thuật:
#   - One-shot final evaluation.
# ============================================================

# Test loader dùng toàn bộ test set.
test_loader = make_loader(
    test_dataset,
    shuffle=False,
)

y_true_test, y_prob_test = predict_loader(
    best_model,
    test_loader,
)

test_metrics = compute_metrics(
    y_true_test,
    y_prob_test,
    locked_threshold,
)

print("=" * 72)
print("FINAL TEST RESULT — LOCKED PHASE 2 MODEL")
print("=" * 72)
print(
    f"Threshold : {locked_threshold:.4f}"
)
print(
    f"Accuracy  : {test_metrics['accuracy']:.4f}"
)
print(
    f"Precision : {test_metrics['precision']:.4f}"
)
print(
    f"Recall    : {test_metrics['recall']:.4f}"
)
print(
    f"F1-score  : {test_metrics['f1']:.4f}"
)
print(
    f"ROC-AUC   : {test_metrics['auc']:.4f}"
)
print("=" * 72)

with open(
    OUTPUT_DIR / "FINAL_TEST_RESULT.json",
    "w",
) as f:
    json.dump(
        {
            "threshold": float(
                locked_threshold
            ),
            **{
                k: float(v)
                for k, v
                in test_metrics.items()
            },
        },
        f,
        indent=2,
    )

print(
    "Test result saved to:",
    OUTPUT_DIR
    / "FINAL_TEST_RESULT.json",
)
